# Reading and Writing Datasets & Metadata

This tutorial shows how `xsnow` handles data I/O and metadata:

1) Read datasets
2) Inspect dimensions, coordinates, and variables
3) View and edit metadata
4) Write datasets to file

`xsnow` focuses on parsing SNOWPACK (PRO/SMET) and CROCUS (NetCDF) and will provide basic readers for CAAML/JSON. It also supports an efficient xsnow NetCDF format for caching large datasets.

- **SNOWPACK**: native PRO and SMET
- **CROCUS**: NetCDF *(planned)*
- **CAAML**: XML/JSON *(planned)*
- **xsnow NetCDF**: optimized for `xsnow`’s data model `xsnowDataset`

```{admonition} Sample datasets
:class: note

`xsnow` ships several sample datasets. Two lightweight ones are available via `xsnow.single_profile()` and `xsnow.single_profile_timeseries()`. More datasets live in [`xsnow.sample_data`](../api/_generated/xsnow.sample_data) (see API docs).
```

## 1. Loading example SNOWPACK datasets
For this SNOWPACK section, we use a sample dataset of gridded simulations shipped in native SNOWPACK formats. Get the sample data path:

In [1]:
import xsnow
datapath = xsnow.sample_data.snp_gridded_dir()

The function returns a directory path. Its structure looks like:

In [2]:
# cell hidden through metadata
import os
print(f"Data location: {datapath}")
for root, dirs, files in os.walk(datapath):
    level = root.replace(datapath, "").count(os.sep)
    indent = " " * 4 * level
    print(f"{indent}{os.path.basename(root)}/")
    subindent = " " * 4 * (level + 1)
    fcounter = 0
    for f in sorted(files):
        if fcounter < 3 or fcounter > len(files)-3:
            print(f"{subindent}{f}")
        elif fcounter == 3:
            print(f"{subindent}...")
        fcounter += 1


Data location: /home/flo/.cache/xsnow-snp-gridded
xsnow-snp-gridded/
    pros/
        gridded/
            VIR1A.pro
            VIR1A.smet
            VIR1A1.pro
            ...
            VIR5A4.pro
            VIR5A4.smet
    smets/
        gridded/
            forecast/
                VIR1A.smet
                VIR2A.smet
                VIR3A.smet
                VIR4A.smet
                VIR5A.smet
            nowcast/
                VIR1A.smet
                VIR2A.smet
                VIR3A.smet
                VIR4A.smet
                VIR5A.smet


### 1.1 Read a single file
Read a single **PRO** file:

In [3]:
ds = xsnow.read(f"{datapath}/pros/gridded/VIR1A.pro")
print(ds)

<xsnowDataset>
  Locations:  1
  Timestamps: 373 (2024-01-18--2024-02-02)
  Profiles:   NotImplementedYet

    employing the <xarray.Dataset> Size: 488kB
    Dimensions:                   (location: 1, time: 373, slope: 1,
                                   realization: 1, layer: 12)
    Coordinates:
      * location                  (location) int64 8B 0
      * time                      (time) datetime64[ns] 3kB 2024-01-18 ... 2024-0...
      * slope                     (slope) int64 8B 0
      * realization               (realization) int64 8B 0
      * layer                     (layer) int64 96B 0 1 2 3 4 5 6 7 8 9 10 11
        latitude                  (location) float64 8B 47.15
        longitude                 (location) float64 8B 11.19
        altitude                  (location) float64 8B 2.372e+03
        slope_inclination         (slope) float64 8B 0.0
        slope_azimuth             (slope) float64 8B 0.0
        z                         (location, time, slope, reali

The summary shows 1 location and 373 timestamps. `xsnowDataset` wraps an underlying `xarray.Dataset` with additional dimensions (`slope`, `realization`, `layer`).

Read a single **SMET** file (no `layer` dimension):

In [ ]:
# Read a single smet file
ds = xsnow.read(f"{datapath}/pros/gridded/VIR1A.smet")
print(ds)

<xsnowDataset>
  Locations:  1
  Timestamps: 416 (2024-01-16--2024-02-02)
  Profiles:   NotImplementedYet

    employing the <xarray.Dataset> Size: 213kB
    Dimensions:               (location: 1, slope: 1, realization: 1, time: 416)
    Coordinates:
      * location              (location) int64 8B 0
      * slope                 (slope) int64 8B 0
      * realization           (realization) int64 8B 0
      * time                  (time) datetime64[ns] 3kB 2024-01-16T05:00:00 ... 2...
        latitude              (location) float64 8B 47.15
        longitude             (location) float64 8B 11.19
        altitude              (location) float64 8B 2.372e+03
        slope_inclination     (slope) float64 8B nan
        slope_azimuth         (slope) float64 8B nan
    Data variables: (12/63)
        Qs                    (location, slope, realization, time) float64 3kB -3...
        Ql                    (location, slope, realization, time) float64 3kB -3...
        Qg               

### 1.2 Read multiple files
Read multiple files by (1) passing a list of paths or (2) pointing to a directory. Files are merged on shared coordinates (e.g., `location`, `time`). Note that for parsing SNOWPACK formats, the merging is based on identical `StationName`s and `station_name`s (properties of PRO and SMET files). Therefore, please don't use the same station names for different locations.

In [ ]:
# Reading a list of filepaths
ds = xsnow.read([
    f"{datapath}/pros/gridded/VIR1A.pro",            # profile data
    f"{datapath}/pros/gridded/VIR1A.smet",           # scalar output data
    f"{datapath}/smets/gridded/nowcast/VIR1A.smet",  # weather input data
])

In [46]:
# Reading all files within one directory
ds = xsnow.read(f"{datapath}/pros/gridded/")


### 1.3 Add SMET files to an existing dataset
Add `.smet` files to your `xsnowDataset` after reading it with `xsnowDataset.merge_smet()`. You can pass a single file, a list of files, or a directory.

In [47]:
ds = xsnow.read(f"{datapath}/pros/gridded/VIR1A.pro")
ds = ds.merge_smet(f"{datapath}/pros/gridded/VIR1A.smet")

```{tip}
Use `var_prefix` to avoid name collisions (e.g., input SMET variables as `input_TA`, `input_PSUM`, …).
This keeps data organized and prevents overwriting variables with identical names across input/output SMETs.
```

## 2. The `xsnowDataset` and its dimensions
Unlike data models that treat each snow profile as an independent object, `xsnow` is built on a gridded data structure organized along several dimensions. This structure leverages `xarray` for efficient vectorized operations, lazy loading, and scalability. Metadata—units, variable descriptions, time zones, and coordinate reference systems—can be embedded to ensure data consistency.

In an `xsnowDataset`, every dimension is strictly one-dimensional (1D). A dimension can have one or more coordinate variables—1D arrays that describe metadata or spatial context (for example, `location` can have `latitude`, `longitude`, `altitude`). This keeps the number of dimensions fixed while supporting use cases like station networks or weather-model grids.

The dimensions of the data model are:

1. **`location`**
   - Specifies the profile location, often with coordinates `latitude`, `longitude`, `easting`, `northing`, `altitude`.

2. **`time`**
   - A point in time for which the profile is valid.
   - `time` is *timezone-naive* (xsnow does not modify the time zones from data sources).

3. **`slope`**
   - Optionally stratify by slope and aspect via coordinates
     - `inclination` (degrees from horizontal)
     - `azimuth` (degrees from north; e.g., 180° = south)
   - If only one combination of inclination and azimuth exists, the dimension has length 1 (trivial case).

4. **`realization`**
   - Different optional instances at the same `(location, time, slope)`, e.g.,
     - Ensemble weather inputs
     - Alternative model configurations
     - Observed vs. simulated profiles
     - Subgrid surface cover types (e.g., forest vs. open)
   - Length 1 in the trivial case.

5. **`layer`** (snow stratigraphy)
   - Per-layer properties at each `(location, time, slope, realization)`, e.g.,
     - Depth or height information expressed as `z` (zero at snow surface; negative downward, positive upward)
     - Layer thickness, density, grain size, temperature, …
   - Categorical variables (e.g., grain types) are encoded numerically for efficiency; you can access labels via accessors (see Tutorial).

At a specific *location* and *time*, you may have one or more *realizations* for each *slope*. Each realization represents a complete or partial snow profile. Some variables are scalars per profile (e.g., total snow height), while others are per-layer.

| **Type**                           | **Dimensions**                                      |
|-----------------------------------|-----------------------------------------------------|
| Scalar data variable              | `(location, time, slope, realization)`              |
| Layer property (per-snow layer)   | `(location, time, slope, realization, layer)`       |



## 2. Inspect structure and metadata
You can use standard xarray accessors to explore shape and metadata:

In [20]:
# List dimensions and sizes
ds.sizes

Frozen({'location': 1, 'time': 373, 'slope': 1, 'realization': 1, 'layer': 12})

In [ ]:
# List coordinates and their info
ds.coords

Coordinates:
  * location           (location) int64 8B 0
  * time               (time) datetime64[ns] 3kB 2024-01-18 ... 2024-02-02T12...
  * slope              (slope) int64 8B 0
  * realization        (realization) int64 8B 0
  * layer              (layer) int64 96B 0 1 2 3 4 5 6 7 8 9 10 11
    latitude           (location) float64 8B 47.15
    longitude          (location) float64 8B 11.19
    altitude           (location) float64 8B 2.372e+03
    slope_inclination  (slope) float64 8B 0.0
    slope_azimuth      (slope) float64 8B 0.0
    z                  (location, time, slope, realization, layer) float32 18kB ...

In [ ]:
# List all data variables
ds.data_vars

Inspect individual variables (e.g., snow density):

In [9]:
print(ds['density'])

<xarray.DataArray 'density' (location: 1, time: 373, slope: 1, realization: 1,
                             layer: 12)> Size: 18kB
array([[[[[100.2,   nan,   nan, ...,   nan,   nan,   nan]]],


        [[[100.3,   nan,   nan, ...,   nan,   nan,   nan]]],


        [[[100.3,   nan,   nan, ...,   nan,   nan,   nan]]],


        ...,


        [[[213.9, 209. , 223.7, ..., 171.9, 142.4, 124.9]]],


        [[[214. , 209.2, 223.7, ..., 172.6, 143. , 125. ]]],


        [[[214.2, 209.3, 223.8, ..., 173.3, 143.6, 125. ]]]]],
      shape=(1, 373, 1, 1, 12), dtype=float32)
Coordinates:
  * location           (location) int64 8B 0
  * time               (time) datetime64[ns] 3kB 2024-01-18 ... 2024-02-02T12...
  * slope              (slope) int64 8B 0
  * realization        (realization) int64 8B 0
  * layer              (layer) int64 96B 0 1 2 3 4 5 6 7 8 9 10 11
    latitude           (location) float64 8B 47.15
    longitude          (location) float64 8B 11.19
    altitude           (locatio

Each variable carries **attributes (metadata)** like units and descriptions.

View dataset- and variable-level attributes:

In [ ]:
# dataset attributes
ds.attrs

{'Conventions': 'CF-1.8',
 'title': 'Snowpack Profile for VIR1A',
 'source': 'SNOWPACK model output (.pro file)',
 'history': '2025-10-14T03:17:38.371405+00:00: Created by xsnow.',
 'crs': 'EPSG:4326'}

In [ ]:
# variable-level attributes
ds['density'].attrs

{'standard_name': 'mass_density_of_snow',
 'units': 'kg m-3',
 'long_name': 'mass density of snow'}

## 3. Add or modify metadata

Metadata are stored under `.attrs`. You can freely edit or add entries.

For example, add a comment:

In [14]:
ds.attrs['comments'] = 'You can document your thoughts!'
ds.attrs

{'Conventions': 'CF-1.8',
 'title': 'Snowpack Profile for VIR1A',
 'source': 'SNOWPACK model output (.pro file)',
 'history': '2025-10-14T03:17:38.371405+00:00: Created by xsnow.',
 'crs': 'EPSG:4326',
 'comments': 'You can document your thoughts!'}

Modify variable-level attributes, e.g., change units or add remarks:

In [15]:
ds['density'].attrs['comment'] = "What's the density of ice cream?"
ds['density'].attrs

{'standard_name': 'mass_density_of_snow',
 'units': 'kg m-3',
 'long_name': 'mass density of snow',
 'comment': "What's the density of ice cream?"}

It’s good practice to record processing steps in a `history` attribute:

In [ ]:
from datetime import datetime
history_line = f"{datetime.now().isoformat(timespec='seconds')}: modified metadata interactively"
ds.attrs['history'] = ds.attrs.get('history', '') + '\n' + history_line
print(ds.attrs['history'])

2025-10-14T03:17:38.371405+00:00: Created by xsnow.
2025-10-14T05:31:01: modified metadata interactively


## 4. Write datasets to file

You can save your `xsnowDataset` back to file(s). To learn about supported formats, explore the `to_*()` methods in [xsnow_io](../api/_generated/xsnow.xsnow_io) (API docs). For example,

- `to_netcdf()`: Basically an exact dump of the `xsnowDataset`.
- `to_pro()` and `to_smet()`: For every (location, time, slope, realization) you can write one `.smet` and `.pro` file.

First, create a temporary directory to write to:

In [ ]:
import tempfile
tmpdir = tempfile.mkdtemp(prefix="xsnow-demo-")

### 4.1 Write to .pro / .smet

In [ ]:
# Write a single profile timeseries to .pro/.smet
ds.to_pro(f"{tmpdir}/test.pro")
ds.to_smet(f"{tmpdir}/test.smet")

Dataset successfully saved to /tmp/xsnow-demo-7e21tf0a/test.pro
Dataset successfully saved to /tmp/xsnow-demo-7e21tf0a/test.smet


### 4.3 Dump the entire dataset to NetCDF

In [25]:
ds.to_netcdf(f"{tmpdir}/test.nc")

TypeError: Invalid value for attr 'mapping': {0: 'VIR1A'}. For serialization to netCDF files, its value must be of one of the following types: str, Number, ndarray, number, list, tuple, bytes